In [ ]:
import pickle
import numpy as np
import librosa
from modules.HMM import continueHMM
from modules.preprocessing import extract_features
from sklearn.preprocessing import StandardScaler
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import librosa.display as dsp

# ===== 1. Định nghĩa hàm load model (từ main.ipynb) =====
def load_model(model_path):
    """Load HMM models từ file pickle"""
    with open(model_path, 'rb') as f:
        data = pickle.load(f)
    
    models = []
    for params in data['models']:
        model = continueHMM(
            A=params['A'],
            means=params['means'],
            covariances=params['covariances'],
            pi=params['pi']
        )
        models.append(model)
    
    return models, data['class_names'], data['scaler']

# ===== 2. Load model và scaler =====
model_path = './HPMR/models/continue_hmm.pkl'
models_loaded, class_names_loaded, scaler = load_model(model_path)
print(f"✅ Đã load {len(models_loaded)} models: {class_names_loaded}")

# ===== 3. Upload file WAV từ máy tính (Google Colab) =====
from google.colab import files
print("\n📂 Vui lòng upload file WAV của bạn:")
uploaded = files.upload()

# Lấy tên file đầu tiên được upload
audio_file = list(uploaded.keys())[0]
print(f"\n🎵 File được upload: {audio_file}")

# ===== 4. Trích xuất features =====
mfcc_features = extract_features(audio_file)

if mfcc_features is None:
    print("❌ Không thể trích xuất features từ file này!")
else:
    print(f"📊 Shape của MFCC features (raw): {mfcc_features.shape}")

    # Chuẩn hóa bằng scaler đã fit trên train set
    mfcc_features_scaled = scaler.transform(mfcc_features)
    print(f"📊 Shape sau chuẩn hóa: {mfcc_features_scaled.shape}")

    # Đọc audio để hiển thị
    y, sr = librosa.load(audio_file, sr=22050)
    print("\n🔊 Phát audio:")
    display(Audio(y, rate=sr))

    # ===== 5. Dự đoán bằng HMM =====
    print("\n📈 Tính toán log probability cho mỗi class:")
    log_probs = []
    for i, model in enumerate(models_loaded):
        log_prob = model.forward(mfcc_features_scaled)[0]
        log_probs.append(log_prob)
        print(f"   Class {class_names_loaded[i]}: log_prob = {log_prob:.2f}")

    # Chọn class có log_prob cao nhất
    predicted_idx = int(np.argmax(log_probs))
    predicted_class = class_names_loaded[predicted_idx]

    # Tính confidence (xác suất tương đối)
    log_probs_arr = np.array(log_probs)
    probs_normalized = np.exp(log_probs_arr - np.max(log_probs_arr))
    probs_normalized /= probs_normalized.sum()
    confidence = probs_normalized[predicted_idx]

    # ===== 6. Hiển thị kết quả =====
    print("\n" + "="*60)
    print(f"🎯 KẾT QUẢ DỰ ĐOÁN")
    print("="*60)
    print(f"   Số dự đoán: {predicted_class}")
    print(f"   Độ tin cậy: {confidence:.2%}")
    print("="*60)

    # Hiển thị xác suất của tất cả các class
    print("\n📊 Phân phối xác suất:")
    for i, class_name in enumerate(class_names_loaded):
        bar = "█" * int(probs_normalized[i] * 50)
        print(f"   {class_name}: {probs_normalized[i]:>6.1%} {bar}")

    # ===== 7. Vẽ biểu đồ MFCC =====
    plt.figure(figsize=(12, 6))
    
    plt.subplot(2, 1, 1)
    dsp.waveshow(y, sr=sr, alpha=0.6)
    plt.title(f'Waveform - Predicted: {predicted_class} (Confidence: {confidence:.1%})')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')

    plt.subplot(2, 1, 2)
    dsp.specshow(mfcc_features_scaled.T, sr=sr, hop_length=256, x_axis='time', cmap='coolwarm')
    plt.colorbar(format='%+2.0f')
    plt.title('MFCC Features (Normalized)')
    plt.xlabel('Time (s)')
    plt.ylabel('MFCC Coefficients')
    plt.tight_layout()
    plt.show()